# Table 1 Cohort Descriptors: Initial Rhythm and Arrest Location

This notebook generates descriptive rows for manuscript Table 1:

- non-shockable rhythm percentage for eICU and PMAP;
- MIMIC-IV rhythm reported as `NR` because a documented initial arrest rhythm is not reliably available;
- OHCA/IHCA or ED-presenting/procedure-coded composition for each observational cohort.

The notebook intentionally uses conservative labels. For eICU and PMAP, location is inferred from admission/ED fields and should be described as an ED-presenting/OHCA proxy unless manually validated against source documentation. For MIMIC-IV, `CA_ED.csv` and `CA_time.csv` are used when available to distinguish ED-presenting cardiac arrest from procedure-coded CPR events.

In [1]:
import os
import re
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 120)
pd.set_option("display.max_rows", 120)

In [2]:
ROOT = Path.cwd()
if not (ROOT / "eICU").exists() and (ROOT.parent / "eICU").exists():
    ROOT = ROOT.parent

OUT = ROOT / "summarized_results" / "table1_data_pulls"
OUT.mkdir(parents=True, exist_ok=True)

PATHS = {
    "eICU": [
        ROOT / "eICU" / "eICUPredictorsDiag.csv",
        ROOT / "eICU" / "eICUPredictors.csv",
        ROOT / "eICU" / "eICUPredictors1.csv",
    ],
    "PMAP": [
        ROOT / "pmap" / "PMAP_Predictors2.csv",
        ROOT / "pmap" / "PMAP_Predictors.csv",
    ],
    "MIMIC-IV": [
        ROOT / "mimiciv" / "MIMIC_Predictors.csv",
        ROOT / "mimiciv" / "MIMIC_Predictors1.csv",
    ],
}

MIMIC_CA_ED = Path(os.environ.get("MIMIC_CA_ED_CSV", str(ROOT / "mimiciv" / "CA_ED.csv")))
MIMIC_CA_TIME = Path(os.environ.get("MIMIC_CA_TIME_CSV", str(ROOT / "mimiciv" / "CA_time.csv")))

print("Repository root:", ROOT)
print("Output directory:", OUT)

Repository root: /home/mbranda1/ttmhte
Output directory: /home/mbranda1/ttmhte/summarized_results/table1_data_pulls


In [3]:
def first_existing(paths):
    for path in paths:
        if Path(path).exists():
            return Path(path)
    return None


def load_csv(label, paths):
    path = first_existing(paths)
    if path is None:
        print(f"{label}: no predictor CSV found in candidates:")
        for candidate in paths:
            print("  -", candidate)
        return pd.DataFrame(), None
    df = pd.read_csv(path)
    print(f"{label}: loaded {path} with shape {df.shape}")
    return df, path


def find_col(df, candidates):
    lower = {c.lower(): c for c in df.columns}
    for cand in candidates:
        if cand.lower() in lower:
            return lower[cand.lower()]
    return None


def boolish(series):
    if series is None:
        return pd.Series(dtype=float)
    if pd.api.types.is_numeric_dtype(series):
        return pd.to_numeric(series, errors="coerce").fillna(0) != 0
    s = series.astype(str).str.strip().str.lower()
    return s.isin(["1", "true", "t", "yes", "y", "present", "positive"])


def summarize_binary(dataset, variable, values, source, denominator_note="analysis cohort"):
    values = pd.Series(values)
    denom = int(values.notna().sum())
    num = int(values.fillna(False).astype(bool).sum()) if denom else 0
    pct = 100 * num / denom if denom else np.nan
    return {
        "dataset": dataset,
        "variable": variable,
        "n": num,
        "denominator": denom,
        "percent": pct,
        "formatted": "NR" if denom == 0 else f"{num}/{denom} ({pct:.1f}%)",
        "source": source,
        "denominator_note": denominator_note,
    }

## Load Predictor Cohorts

In [4]:
eicu, eicu_path = load_csv("eICU", PATHS["eICU"])
pmap, pmap_path = load_csv("PMAP", PATHS["PMAP"])
mimic, mimic_path = load_csv("MIMIC-IV", PATHS["MIMIC-IV"])

for label, df in [("eICU", eicu), ("PMAP", pmap), ("MIMIC-IV", mimic)]:
    if not df.empty:
        print(f"\n{label} columns containing rhythm/location keywords:")
        cols = [c for c in df.columns if re.search(r"rhythm|rythm|vtach|vfib|asystole|pea|\bvf\b|ed_visit|admit|source|origin|location", c, re.I)]
        print(cols[:120])

/local/mbranda1/3981445/ipykernel_4118204/424301802.py:15: DtypeWarning: Columns (2188,2190) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(path)


eICU: loaded /home/mbranda1/ttmhte/eICU/eICUPredictorsDiag.csv with shape (3172, 2914)
PMAP: loaded /home/mbranda1/ttmhte/pmap/PMAP_Predictors2.csv with shape (1741, 22663)
MIMIC-IV: loaded /home/mbranda1/ttmhte/mimiciv/MIMIC_Predictors.csv with shape (743, 7886)

eICU columns containing rhythm/location keywords:
['hospitaladmittime24', 'hospitaladmitsource', 'treatment_antiarrhythmics', 'treatment_arrhythmias', 'treatment_class I antiarrhythmic', 'treatment_class II antiarrhythmic', 'treatment_class III antiarrhythmic', 'treatment_class IV antiarrhythmic', 'diagnosis_arrhythmias', 'diagnosis_genitourinary source', 'diagnosis_initial rhythm: asystole', 'diagnosis_initial rhythm: pulseless electrical activity', 'diagnosis_initial rhythm: unknown', 'diagnosis_initial rhythm: ventricular fibrillation', 'diagnosis_initial rhythm: ventricular tachycardia', 'diagnosis_likely cardiac origin', 'diagnosis_respiratory source', 'diagnosis_skin or wound source', 'diagnosis_unclear source', 'diagno

## Non-Shockable Rhythm Row for Table 1

In [5]:
rhythm_rows = []
rhythm_components = []

if not eicu.empty:
    vt_col = find_col(eicu, ["VTachy", "diagnosis_initial rhythm: ventricular tachycardia", "diagnosis_ventricular tachycardia"])
    vf_col = find_col(eicu, ["VFib", "diagnosis_initial rhythm: ventricular fibrillation", "diagnosis_ventricular fibrillation"])
    asys_col = find_col(eicu, ["Asystole", "diagnosis_initial rhythm: asystole"])
    pea_col = find_col(eicu, ["PEA", "diagnosis_initial rhythm: pulseless electrical activity"])

    asys = boolish(eicu[asys_col]) if asys_col else pd.Series(False, index=eicu.index)
    pea = boolish(eicu[pea_col]) if pea_col else pd.Series(False, index=eicu.index)
    vt = boolish(eicu[vt_col]) if vt_col else pd.Series(False, index=eicu.index)
    vf = boolish(eicu[vf_col]) if vf_col else pd.Series(False, index=eicu.index)
    any_rhythm = asys | pea | vt | vf
    nonshock = asys | pea

    rhythm_rows.append(summarize_binary(
        "eICU-CRD",
        "Non-shockable initial rhythm (asystole/PEA)",
        nonshock,
        source=f"{eicu_path}; columns asystole={asys_col}, pea={pea_col}, vt={vt_col}, vf={vf_col}",
    ))
    rhythm_components.extend([
        summarize_binary("eICU-CRD", "Asystole", asys, f"{eicu_path}; {asys_col}"),
        summarize_binary("eICU-CRD", "PEA", pea, f"{eicu_path}; {pea_col}"),
        summarize_binary("eICU-CRD", "VT", vt, f"{eicu_path}; {vt_col}"),
        summarize_binary("eICU-CRD", "VF", vf, f"{eicu_path}; {vf_col}"),
        summarize_binary("eICU-CRD", "Any documented rhythm flag", any_rhythm, f"{eicu_path}"),
    ])

if not pmap.empty:
    asys_col = find_col(pmap, ["asystole", "cardiac asystole", "asystole by electrocardiogram"])
    pea_col = find_col(pmap, ["pea", "pulseless electrical activity", "cardiac arrest with pulseless electrical activity"])
    vf_col = find_col(pmap, ["VF", "ventricular fibrillation", "cardiac arrest with ventricular fibrillation"])

    asys = boolish(pmap[asys_col]) if asys_col else pd.Series(False, index=pmap.index)
    pea = boolish(pmap[pea_col]) if pea_col else pd.Series(False, index=pmap.index)
    vf = boolish(pmap[vf_col]) if vf_col else pd.Series(False, index=pmap.index)
    any_rhythm = asys | pea | vf
    nonshock = asys | pea

    rhythm_rows.append(summarize_binary(
        "PMAP",
        "Non-shockable initial rhythm (asystole/PEA)",
        nonshock,
        source=f"{pmap_path}; columns asystole={asys_col}, pea={pea_col}, vf={vf_col}",
    ))
    rhythm_components.extend([
        summarize_binary("PMAP", "Asystole", asys, f"{pmap_path}; {asys_col}"),
        summarize_binary("PMAP", "PEA", pea, f"{pmap_path}; {pea_col}"),
        summarize_binary("PMAP", "VF", vf, f"{pmap_path}; {vf_col}"),
        summarize_binary("PMAP", "Any documented rhythm flag", any_rhythm, f"{pmap_path}"),
    ])

rhythm_rows.append({
    "dataset": "MIMIC-IV",
    "variable": "Non-shockable initial rhythm (asystole/PEA)",
    "n": np.nan,
    "denominator": len(mimic) if not mimic.empty else np.nan,
    "percent": np.nan,
    "formatted": "NR",
    "source": "No reliable documented initial arrest rhythm in current MIMIC-IV extraction; ICD VF proxy only.",
    "denominator_note": "not reported",
})

rhythm_table = pd.DataFrame(rhythm_rows)
rhythm_component_table = pd.DataFrame(rhythm_components)
display(rhythm_table)
display(rhythm_component_table)
rhythm_table.to_csv(OUT / "table1_nonshockable_rhythm_row.csv", index=False)
rhythm_component_table.to_csv(OUT / "rhythm_component_counts.csv", index=False)

,dataset,variable,n,denominator,percent,formatted,source,denominator_note
0,eICU-CRD,Non-shockable initial rhythm (asystole/PEA),660.0,3172,20.807062,660/3172 (20.8%),/home/mbranda1/ttmhte/eICU/eICUPredictorsDiag....,analysis cohort
1,PMAP,Non-shockable initial rhythm (asystole/PEA),138.0,1741,7.926479,138/1741 (7.9%),/home/mbranda1/ttmhte/pmap/PMAP_Predictors2.cs...,analysis cohort
2,MIMIC-IV,Non-shockable initial rhythm (asystole/PEA),NaN,743,NaN,NR,No reliable documented initial arrest rhythm i...,not reported


,dataset,variable,n,denominator,percent,formatted,source,denominator_note
0,eICU-CRD,Asystole,247,3172,7.786885,247/3172 (7.8%),/home/mbranda1/ttmhte/eICU/eICUPredictorsDiag....,analysis cohort
1,eICU-CRD,PEA,424,3172,13.366961,424/3172 (13.4%),/home/mbranda1/ttmhte/eICU/eICUPredictorsDiag....,analysis cohort
2,eICU-CRD,VT,116,3172,3.656999,116/3172 (3.7%),/home/mbranda1/ttmhte/eICU/eICUPredictorsDiag....,analysis cohort
3,eICU-CRD,VF,274,3172,8.638083,274/3172 (8.6%),/home/mbranda1/ttmhte/eICU/eICUPredictorsDiag....,analysis cohort
4,eICU-CRD,Any documented rhythm flag,1024,3172,32.282472,1024/3172 (32.3%),/home/mbranda1/ttmhte/eICU/eICUPredictorsDiag.csv,analysis cohort
5,PMAP,Asystole,41,1741,2.354968,41/1741 (2.4%),/home/mbranda1/ttmhte/pmap/PMAP_Predictors2.cs...,analysis cohort
6,PMAP,PEA,97,1741,5.571511,97/1741 (5.6%),/home/mbranda1/ttmhte/pmap/PMAP_Predictors2.cs...,analysis cohort
7,PMAP,VF,51,1741,2.929351,51/1741 (2.9%),/home/mbranda1/ttmhte/pmap/PMAP_Predictors2.cs...,analysis cohort
8,PMAP,Any documented rhythm flag,188,1741,10.798392,188/1741 (10.8%),/home/mbranda1/ttmhte/pmap/PMAP_Predictors2.csv,analysis cohort


## OHCA/IHCA or ED-Presenting Composition

In [6]:
location_rows = []
location_detail_rows = []

if not eicu.empty:
    source_col = find_col(eicu, ["hospitaladmitsource", "unitadmitsource", "admission_source"])
    if source_col:
        src = eicu[source_col].astype(str).str.strip()
        # Conservative proxy: ED admissions are treated as ED-presenting/OHCA proxy.
        ohca_proxy = src.str.contains(r"emergency|\bed\b", case=False, regex=True, na=False)
        known = src.ne("") & src.str.lower().ne("nan")
        location_rows.append(summarize_binary(
            "eICU-CRD",
            "ED-presenting/OHCA proxy",
            ohca_proxy.where(known, np.nan),
            source=f"{eicu_path}; {source_col}",
            denominator_note="known hospital admit source",
        ))
        location_rows.append(summarize_binary(
            "eICU-CRD",
            "Non-ED/IHCA proxy",
            (~ohca_proxy).where(known, np.nan),
            source=f"{eicu_path}; {source_col}",
            denominator_note="known hospital admit source",
        ))
        counts = src.value_counts(dropna=False).rename_axis("hospital_admit_source").reset_index(name="n")
        counts["dataset"] = "eICU-CRD"
        location_detail_rows.append(counts)
    else:
        location_rows.append({"dataset": "eICU-CRD", "variable": "OHCA/IHCA composition", "formatted": "NR", "source": "No admission source column found."})

if not pmap.empty:
    ed_col = find_col(pmap, ["ed_visit_yn"])
    if ed_col:
        ed = boolish(pmap[ed_col])
        location_rows.append(summarize_binary(
            "PMAP",
            "ED-presenting/OHCA proxy",
            ed,
            source=f"{pmap_path}; {ed_col}",
            denominator_note="analysis cohort",
        ))
        location_rows.append(summarize_binary(
            "PMAP",
            "Non-ED/IHCA proxy",
            ~ed,
            source=f"{pmap_path}; {ed_col}",
            denominator_note="analysis cohort",
        ))
        counts = pmap[ed_col].value_counts(dropna=False).rename_axis("ed_visit_yn").reset_index(name="n")
        counts["dataset"] = "PMAP"
        location_detail_rows.append(counts)
    else:
        location_rows.append({"dataset": "PMAP", "variable": "OHCA/IHCA composition", "formatted": "NR", "source": "No ed_visit_yn column found."})

def load_optional(path):
    path = Path(path)
    if path.exists():
        df = pd.read_csv(path)
        print("Loaded", path, df.shape)
        return df
    print("Missing optional file:", path)
    return pd.DataFrame()

ca_ed = load_optional(MIMIC_CA_ED)
ca_time = load_optional(MIMIC_CA_TIME)
if not mimic.empty:
    mimic_ids = mimic[[c for c in ["subject_id", "hadm_id"] if c in mimic.columns]].drop_duplicates()
    if "hadm_id" in mimic_ids.columns and not ca_ed.empty and not ca_time.empty:
        ed_hadm = set(pd.to_numeric(ca_ed.get("hadm_id"), errors="coerce").dropna().astype(int))
        time_hadm = set(pd.to_numeric(ca_time.get("hadm_id"), errors="coerce").dropna().astype(int))
        hadm = pd.to_numeric(mimic_ids["hadm_id"], errors="coerce")
        ed_presenting = hadm.isin(ed_hadm)
        procedure_coded_only = hadm.isin(time_hadm - ed_hadm)
        classified = ed_presenting | procedure_coded_only
        location_rows.append(summarize_binary(
            "MIMIC-IV",
            "ED-presenting/OHCA source",
            ed_presenting.where(classified, np.nan),
            source=f"{MIMIC_CA_ED.name} vs {MIMIC_CA_TIME.name}; hadm_id overlap",
            denominator_note="cohort admissions classifiable from CA_ED/CA_time",
        ))
        location_rows.append(summarize_binary(
            "MIMIC-IV",
            "Procedure-coded CPR/IHCA source",
            procedure_coded_only.where(classified, np.nan),
            source=f"{MIMIC_CA_ED.name} vs {MIMIC_CA_TIME.name}; hadm_id overlap",
            denominator_note="cohort admissions classifiable from CA_ED/CA_time",
        ))
    elif not ca_ed.empty and not ca_time.empty and "subject_id" in mimic_ids.columns:
        ed_subj = set(pd.to_numeric(ca_ed.get("subject_id"), errors="coerce").dropna().astype(int))
        time_subj = set(pd.to_numeric(ca_time.get("subject_id"), errors="coerce").dropna().astype(int))
        subj = pd.to_numeric(mimic_ids["subject_id"], errors="coerce")
        ed_presenting = subj.isin(ed_subj)
        procedure_coded_only = subj.isin(time_subj - ed_subj)
        classified = ed_presenting | procedure_coded_only
        location_rows.append(summarize_binary(
            "MIMIC-IV",
            "ED-presenting/OHCA source",
            ed_presenting.where(classified, np.nan),
            source=f"{MIMIC_CA_ED.name} vs {MIMIC_CA_TIME.name}; subject_id overlap",
            denominator_note="cohort subjects classifiable from CA_ED/CA_time",
        ))
        location_rows.append(summarize_binary(
            "MIMIC-IV",
            "Procedure-coded CPR/IHCA source",
            procedure_coded_only.where(classified, np.nan),
            source=f"{MIMIC_CA_ED.name} vs {MIMIC_CA_TIME.name}; subject_id overlap",
            denominator_note="cohort subjects classifiable from CA_ED/CA_time",
        ))
    else:
        location_rows.append({
            "dataset": "MIMIC-IV",
            "variable": "OHCA/IHCA composition",
            "n": np.nan,
            "denominator": len(mimic),
            "percent": np.nan,
            "formatted": "NR",
            "source": "Need CA_ED.csv and CA_time.csv to classify ED-presenting vs procedure-coded source.",
            "denominator_note": "not reported",
        })

location_table = pd.DataFrame(location_rows)
display(location_table)
location_table.to_csv(OUT / "table1_ohca_ihca_composition.csv", index=False)

if location_detail_rows:
    location_detail = pd.concat(location_detail_rows, ignore_index=True, sort=False)
    display(location_detail)
    location_detail.to_csv(OUT / "location_source_value_counts.csv", index=False)

Loaded /home/mbranda1/ttmhte/mimiciv/CA_ED.csv (531, 9)
Loaded /home/mbranda1/ttmhte/mimiciv/CA_time.csv (500, 11)


/local/mbranda1/3981445/ipykernel_4118204/424301802.py:40: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  num = int(values.fillna(False).astype(bool).sum()) if denom else 0
/local/mbranda1/3981445/ipykernel_4118204/424301802.py:40: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  num = int(values.fillna(False).astype(bool).sum()) if denom else 0


,dataset,variable,n,denominator,percent,formatted,source,denominator_note
0,eICU-CRD,ED-presenting/OHCA proxy,1470,2329,63.117218,1470/2329 (63.1%),/home/mbranda1/ttmhte/eICU/eICUPredictorsDiag....,known hospital admit source
1,eICU-CRD,Non-ED/IHCA proxy,859,2329,36.882782,859/2329 (36.9%),/home/mbranda1/ttmhte/eICU/eICUPredictorsDiag....,known hospital admit source
2,PMAP,ED-presenting/OHCA proxy,1489,1741,85.525560,1489/1741 (85.5%),/home/mbranda1/ttmhte/pmap/PMAP_Predictors2.cs...,analysis cohort
3,PMAP,Non-ED/IHCA proxy,252,1741,14.474440,252/1741 (14.5%),/home/mbranda1/ttmhte/pmap/PMAP_Predictors2.cs...,analysis cohort
4,MIMIC-IV,ED-presenting/OHCA source,498,743,67.025572,498/743 (67.0%),CA_ED.csv vs CA_time.csv; subject_id overlap,cohort subjects classifiable from CA_ED/CA_time
5,MIMIC-IV,Procedure-coded CPR/IHCA source,245,743,32.974428,245/743 (33.0%),CA_ED.csv vs CA_time.csv; subject_id overlap,cohort subjects classifiable from CA_ED/CA_time


,hospital_admit_source,n,dataset,ed_visit_yn
0,Emergency Department,1470,eICU-CRD,NaN
1,nan,843,eICU-CRD,NaN
2,Floor,312,eICU-CRD,NaN
3,Direct Admit,256,eICU-CRD,NaN
4,Other Hospital,87,eICU-CRD,NaN
5,Acute Care/Floor,72,eICU-CRD,NaN
6,Operating Room,56,eICU-CRD,NaN
7,Step-Down Unit (SDU),43,eICU-CRD,NaN
8,Recovery Room,17,eICU-CRD,NaN
9,Chest Pain Center,7,eICU-CRD,NaN


## Copy-Ready Table Rows

In [7]:
datasets = ["HYPERION", "eICU-CRD", "PMAP", "MIMIC-IV"]

nonshock_map = {"HYPERION": "100.0% by design"}
for _, row in rhythm_table.iterrows():
    nonshock_map[row["dataset"]] = row["formatted"]

# Use the OHCA/ED-presenting row as the primary composition row.
location_map = {"HYPERION": "included OHCA and IHCA; trial non-shockable only"}
if "location_table" in globals() and not location_table.empty:
    for dataset in ["eICU-CRD", "PMAP", "MIMIC-IV"]:
        sub = location_table[(location_table["dataset"] == dataset) & (location_table["variable"].str.contains("OHCA|ED-presenting", case=False, na=False))]
        if len(sub):
            location_map[dataset] = sub.iloc[0]["formatted"]
        else:
            location_map[dataset] = "NR"

copy_ready = pd.DataFrame([
    {"Table 1 row": "Non-shockable initial rhythm (asystole/PEA), n/N (%)", **{d: nonshock_map.get(d, "NR") for d in datasets}},
    {"Table 1 row": "OHCA or ED-presenting arrest, n/N (%)", **{d: location_map.get(d, "NR") for d in datasets}},
])
display(copy_ready)
copy_ready.to_csv(OUT / "copy_ready_table1_rows.csv", index=False)

notes = pd.DataFrame([
    {"note": "MIMIC-IV initial rhythm should remain NR because only diagnosis/proxy rhythm signals are available, not a reliable documented initial arrest rhythm."},
    {"note": "eICU and PMAP OHCA/IHCA rows are admission-source/ED-visit proxies unless independently chart-validated."},
    {"note": "For MIMIC-IV location, CA_ED indicates ED-presenting source and CA_time indicates procedure-coded CPR source; admissions in both files are classified as ED-presenting."},
])
display(notes)
notes.to_csv(OUT / "table1_data_pull_notes.csv", index=False)

,Table 1 row,HYPERION,eICU-CRD,PMAP,MIMIC-IV
0,"Non-shockable initial rhythm (asystole/PEA), n...",100.0% by design,660/3172 (20.8%),138/1741 (7.9%),NR
1,"OHCA or ED-presenting arrest, n/N (%)",included OHCA and IHCA; trial non-shockable only,1470/2329 (63.1%),1489/1741 (85.5%),498/743 (67.0%)


,note
0,MIMIC-IV initial rhythm should remain NR becau...
1,eICU and PMAP OHCA/IHCA rows are admission-sou...
2,"For MIMIC-IV location, CA_ED indicates ED-pres..."
